<a href="https://colab.research.google.com/github/siamliam12/deep-learning-scratchpad/blob/main/genomic_attention_modeler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#importing libraries
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch.nn as nn

we are creating a custom pytorch dataset to convert raw DNA strings into one-hot encoded tensors. We store the raw list of biological sequences in ```self.sequences``` the hashmap is the ```self.vocab```

```__len__()``` function:

pytorch needs to know exactly how many sequences exist in the dataset

```__getitem__()``` function:

- grabs a single string based on the index in ```seq = self.sequences[idx]```
- Maps the letters to integers using our vocab
- convert the python list to a pytorch tensor
- finally blast into a 2d orthogonal matrix and convert to float 32 for neural network math

In [2]:
class GenomicDataset(Dataset):
    def __init__(self, sequences):
      self.sequences = sequences
      self.vocab = {'A':0,'C':1,'T':2,'G':3}

    def __len__(self):
      return len(self.sequences)

    def __getitem__(self, idx):
       seq = self.sequences[idx]
       int_seq = [self.vocab[char] for char in seq]
       tensor_seq = torch.tensor(int_seq,dtype=torch.int64)
       one_hot = torch.nn.functional.one_hot(tensor_seq, num_classes=4).to(torch.float32)
       return one_hot

In [3]:
def genomic_collate_fn(batch):
    padded_batch = pad_sequence(batch,batch_first=True,padding_value=0.0)
    return padded_batch

In [4]:
# Dummy test
# 1. Create a dummy list of sequences with mismatched lengths
dummy_data = ["ACTG", "GC", "ATATGC"]

# 2. Instantiate our custom dataset
dataset = GenomicDataset(dummy_data)

# 3. Create the DataLoader, explicitly passing our custom collate function
dataloader = DataLoader(dataset, batch_size=3, shuffle=True, collate_fn=genomic_collate_fn)

# 4. Pull exactly one batch to inspect the pipeline
for batch in dataloader:
    print("Final Batch Shape:", batch.shape)
    print("Batch Data:\n", batch)
    break

Final Batch Shape: torch.Size([3, 6, 4])
Batch Data:
 tensor([[[0., 0., 0., 1.],
         [0., 1., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]],

        [[1., 0., 0., 0.],
         [0., 1., 0., 0.],
         [0., 0., 1., 0.],
         [0., 0., 0., 1.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]],

        [[1., 0., 0., 0.],
         [0., 0., 1., 0.],
         [1., 0., 0., 0.],
         [0., 0., 1., 0.],
         [0., 0., 0., 1.],
         [0., 1., 0., 0.]]])


# Attention layer
- initialize the attention engine. embed_dim=4 because our one-hot vectors have exactly 4 features(A,C,T,G)
- num_heads =1 keeps the math simple
- batch_first=True forces Pytorch to accept our (batch,seq,feature) layout

- generate the key padding mask dynamically
- we sum the values across the innermost feature dimension (dim=-1)
- if the sum is exactly 0.0 it's a padding vector, so we mark it as true.

In [6]:
attention_layer = nn.MultiheadAttention(embed_dim=4,num_heads=1,batch_first=True)
padding_mask = (batch.sum(dim=-1)==0.0)
print("Padding Mask:\n",padding_mask)
# The forward pass
attn_output,attn_weights = attention_layer(
    query=batch,
    key=batch,
    value=batch,
    key_padding_mask=padding_mask
)
print("Attention Output Shape: ",attn_output.shape)
print("Attention Weights Shape: ",attn_weights.shape)

Padding Mask:
 tensor([[False, False,  True,  True,  True,  True],
        [False, False, False, False,  True,  True],
        [False, False, False, False, False, False]])
Attention Output Shape:  torch.Size([3, 6, 4])
Attention Weights Shape:  torch.Size([3, 6, 6])
